# 🤖 Unidad 4 — Reto GAIA (Hugging Face Agents Course)

**Objetivo:** responder las 20 preguntas del reto GAIA con un agente, superar el **30%** (certificado) y apuntar al **70% (14/20)**.

## Cómo usar este cuaderno
1. **Entorno de ejecución → Reiniciar y ejecutar todo** cada vez que lo abras (o Ctrl+F9).
2. Completa tus claves en el Paso 1.
3. Ejecuta los pasos **en orden**: descarga preguntas → ground truth → adjuntos → herramientas → agente.
4. Primero la **prueba de modelos** (5 preguntas), luego las 20 completas con el ganador.
5. Revisa tu nota **en local** (Paso 9) antes de enviar al leaderboard (Paso 10).

> Las 20 preguntas salen del dataset público `gaia-benchmark/GAIA` (validación, nivel 1), así que podemos comparar en local antes de enviar. El scoring oficial es: `respuesta.strip().lower() == ground_truth.strip().lower()`.

## Paso 0 — Instalar librerías

In [ ]:
!pip install -q smolagents litellm datasets pandas openpyxl pypdf ddgs beautifulsoup4 requests

## Paso 1 — Claves API

Pega tus claves entre las comillas. Las obligatorias: **GEMINI** (gratis en aistudio.google.com) o **OPENAI** (la de tu app). **GROQ** es opcional (gratis en console.groq.com) para probar Qwen3-32B. **TAVILY** es opcional pero muy recomendable para buscar en la web (gratis, 1000 búsquedas/mes, sin tarjeta → tavily.com).

**Importante:** para el Paso 3 necesitarás además un **token de Hugging Face** (gratis): acepta las condiciones del dataset en [gaia-benchmark/GAIA](https://huggingface.co/datasets/gaia-benchmark/GAIA) y crea un token de solo lectura en [settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = "PEGA_AQUI_TU_CLAVE_DE_GOOGLE_AI_STUDIO"
os.environ["OPENAI_API_KEY"]  = "PEGA_AQUI_TU_CLAVE_DE_OPENAI"
os.environ["GROQ_API_KEY"]    = ""   # opcional
os.environ["TAVILY_API_KEY"]  = ""   # opcional pero recomendable

# litellm no se queja de parámetros que un modelo no acepta
import litellm
litellm.drop_params = True
print("Claves configuradas.")

## Paso 2 — Descargar las 20 preguntas del reto

In [ ]:
import json, requests, time, os

API = "https://agents-course-unit4-scoring.hf.space"

def obtener_preguntas():
    for base in (API, "https://agents-course-unit4.hf.space"):
        try:
            r = requests.get(base + "/questions", timeout=30)
            if r.status_code == 200:
                return r.json(), base
        except Exception as e:
            print(f"{base}: {e}")
    return None, None

preguntas, API = obtener_preguntas()
assert preguntas, ("No se pudo descargar. Plan B: abre " + API + "/questions en el navegador, "
                   "copia el JSON y pégalo en questions.json en Colab (icono de carpeta → botón derecho).")

with open("questions.json", "w") as f:
    json.dump(preguntas, f, ensure_ascii=False, indent=2)

print(f"{len(preguntas)} preguntas descargadas desde {API}")
for i, q in enumerate(preguntas, 1):
    t = q.get("question", q.get("Question", ""))[:90].replace("\n", " ")
    print(f"{i:2d}. [{q['task_id'][:8]}] {t}...")

## Paso 3 — Verdaderas respuestas (ground truth público)

Cruzamos los `task_id` con el dataset público de GAIA. Esto nos da la respuesta correcta de cada pregunta **para practicar en local**. Deben coincidir las 20.

In [ ]:
from huggingface_hub import login
from datasets import load_dataset

# 1) Acepta las condiciones UNA VEZ en el navegador:
#    https://huggingface.co/datasets/gaia-benchmark/GAIA  → "Agree and access repository"
# 2) Crea un token de solo lectura en: https://huggingface.co/settings/tokens
# 3) Pégalo aquí:
login("PEGAR_TOKEN_AQUI")   # o login() sin argumentos para pegarlo en un cuadro emergente

ds = None
for cfg in ("2023_level1", "2023_all"):
    try:
        ds = load_dataset("gaia-benchmark/GAIA", cfg, split="validation")
        print("Dataset cargado con configuración:", cfg)
        break
    except Exception as e:
        print("Falló", cfg, "→", str(e)[:150])

assert ds is not None, "No se pudo cargar el dataset de GAIA."

ids_curso = {str(q["task_id"]) for q in preguntas}
ground_truth = {}
for ex in ds:
    tid = str(ex.get("task_id"))
    if tid in ids_curso:
        ground_truth[tid] = str(ex.get("Final answer", ""))

with open("gt.json", "w") as f:
    json.dump(ground_truth, f, ensure_ascii=False, indent=2)

print(f"\nCoinciden {len(ground_truth)}/{len(preguntas)} task_id con el dataset público.")
for q in preguntas:
    tid = str(q["task_id"])
    print(" ✅" if tid in ground_truth else " ❌", tid[:8], "→", ground_truth.get(tid, "SIN DATO")[:50])

## Paso 4 — Descargar los archivos adjuntos

Algunas preguntas traen un archivo (xlsx, pdf, imagen, audio...). Los guardamos en `archivos/` para que el agente los lea. Si una pregunta no tiene archivo, el servidor devuelve 404 y la saltamos.

In [ ]:
import os, json, shutil, requests, re

os.makedirs("archivos", exist_ok=True)
adjuntos = {}

# Intento 1: endpoint del curso
for q in preguntas:
    tid = str(q["task_id"])
    try:
        r = requests.get(f"{API}/files/{tid}", timeout=60)
        if r.status_code != 200 or not r.content:
            continue
        nombre = "archivo"
        cd = r.headers.get("Content-Disposition", "")
        m = re.search(r'filename="?([^";]+)', cd)
        if m:
            nombre = m.group(1)
        ruta = os.path.join("archivos", f"{tid[:8]}_{nombre}")
        with open(ruta, "wb") as f:
            f.write(r.content)
        adjuntos[tid] = ruta
    except Exception:
        pass

# Intento 2 (respaldo): archivos originales del dataset público de GAIA
try:
    from huggingface_hub import snapshot_download
    data_dir = snapshot_download("gaia-benchmark/GAIA", repo_type="dataset")
    ids = {str(q["task_id"]) for q in preguntas}
    for ex in ds:
        tid = str(ex.get("task_id"))
        fp = ex.get("file_path")
        if tid in ids and fp and tid not in adjuntos:
            origen = os.path.join(data_dir, fp)
            if os.path.exists(origen):
                ext = os.path.splitext(fp)[1]
                ruta = os.path.join("archivos", f"{tid[:8]}{ext}")
                shutil.copy(origen, ruta)
                adjuntos[tid] = ruta
except Exception as e:
    print("Respaldo no disponible:", str(e)[:120])

with open("adjuntos.json", "w") as f:
    json.dump(adjuntos, f, ensure_ascii=False, indent=2)

print(f"{len(adjuntos)} preguntas tienen archivo adjunto:")
for tid, ruta in adjuntos.items():
    print(" -", tid[:8], "→", ruta)

## Paso 5 — Herramientas del agente

- **buscar_web**: busca en internet (Tavily si tienes clave, si no DuckDuckGo).
- **visitar_url**: lee el contenido de una página web.
- **descargar_archivo**: descarga un adjunto por su URL al disco.
- **leer_archivo**: lee txt/csv/json/xlsx/pdf; las imágenes se describen con Gemini (visión); el audio se transcribe con Groq si tienes clave.

In [ ]:
import os, json, base64, requests, time
from smolagents import tool
from litellm import completion

@tool
def buscar_web(consulta: str) -> str:
    """Busca en internet. Devuelve titulares y fragmentos de las mejores páginas.
    Args:
        consulta: lo que quieres buscar, en inglés suele dar mejores resultados
    """
    tv = os.environ.get("TAVILY_API_KEY", "")
    if tv:
        try:
            r = requests.post("https://api.tavily.com/search",
                              json={"api_key": tv, "query": consulta, "max_results": 5},
                              timeout=30)
            if r.status_code == 200:
                res = r.json().get("results", [])
                if res:
                    return "\n".join(f"- {x.get('title','')}: {x.get('content','')[:400]}" for x in res)
        except Exception:
            pass
    try:
        try:
            from ddgs import DDGS
        except ImportError:
            from duckduckgo_search import DDGS
        with DDGS() as d:
            res = list(d.text(consulta, max_results=5))
            if res:
                return "\n".join(f"- {x.get('title','')}: {x.get('body','')[:400]}" for x in res)
        return "Sin resultados."
    except Exception as e:
        return f"Búsqueda no disponible ({e}). Prueba visitar_url con una URL conocida."

@tool
def visitar_url(url: str) -> str:
    """Lee el contenido (texto) de una página web.
    Args:
        url: dirección completa de la página
    """
    try:
        r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code != 200:
            return f"HTTP {r.status_code} al abrir {url}"
        try:
            from bs4 import BeautifulSoup
            texto = BeautifulSoup(r.text, "html.parser").get_text(" ", strip=True)
        except Exception:
            texto = r.text
        return texto[:8000] + ("\n...[recortado]" if len(texto) > 8000 else "")
    except Exception as e:
        return f"Error visitando {url}: {e}"

@tool
def descargar_archivo(url: str, nombre: str = "descarga") -> str:
    """Descarga un archivo (por ejemplo un adjunto de pregunta) al disco local.
    Args:
        url: URL del archivo
        nombre: nombre con el que guardarlo
    """
    try:
        r = requests.get(url, timeout=120)
        if r.status_code != 200:
            return f"HTTP {r.status_code}"
        ruta = os.path.join("archivos", nombre)
        os.makedirs("archivos", exist_ok=True)
        with open(ruta, "wb") as f:
            f.write(r.content)
        return f"Guardado en {ruta} ({len(r.content)} bytes)."
    except Exception as e:
        return f"Error: {e}"

def _describir_imagen(ruta: str) -> str:
    with open(ruta, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    ext = ruta.rsplit(".", 1)[-1].lower().replace("jpg", "jpeg")
    uri = f"data:image/{ext};base64,{b64}"
    resp = completion(
        model="gemini/gemini-3.5-flash",
        messages=[{"role": "user", "content": [
            {"type": "text", "text": "Describe this image in detail, transcribing any visible text."},
            {"type": "image_url", "image_url": {"url": uri}}]}],
        max_tokens=1500)
    return resp.choices[0].message.content

def _transcribir_audio(ruta: str) -> str:
    gq = os.environ.get("GROQ_API_KEY", "")
    if not gq:
        return "No hay GROQ_API_KEY para transcribir audio."
    import litellm
    with open(ruta, "rb") as f:
        t = litellm.transcription(model="groq/whisper-large-v3", file=f)
    return getattr(t, "text", str(t))

@tool
def leer_archivo(ruta: str) -> str:
    """Lee un archivo local: texto, csv, json, xlsx, pdf; describe imágenes y transcribe audios.
    Args:
        ruta: ruta del archivo, por ejemplo archivos/abc12345_datos.xlsx
    """
    if not os.path.exists(ruta):
        return "No existe esa ruta. Usa descargar_archivo primero o revisa adjuntos.json."
    ext = ruta.rsplit(".", 1)[-1].lower()
    try:
        if ext in ("txt", "md", "csv", "json", "py", "html"):
            with open(ruta, "r", errors="replace") as f:
                return f.read()[:8000]
        if ext == "xlsx":
            import pandas as pd
            xl = pd.ExcelFile(ruta)
            out = [f"Hojas: {xl.sheet_names}"]
            for h in xl.sheet_names[:5]:
                df = xl.parse(h)
                out.append(f"--- Hoja '{h}' ({df.shape[0]} filas x {df.shape[1]} cols) ---")
                out.append(df.head(20).to_string())
            return "\n".join(out)[:9000]
        if ext == "pdf":
            from pypdf import PdfReader
            r = PdfReader(ruta)
            texto = "\n".join((p.extract_text() or "") for p in r.pages[:15])
            return texto[:9000] + ("\n...[recortado]" if len(texto) > 9000 else "")
        if ext in ("png", "jpg", "jpeg", "webp", "gif"):
            return _describir_imagen(ruta)
        if ext in ("mp3", "wav", "m4a", "ogg"):
            return _transcribir_audio(ruta)
        return f"Tipo .{ext} no soportado. Intenta leerlo como texto."
    except Exception as e:
        return f"Error leyendo {ruta}: {e}"

HERRAMIENTAS = [buscar_web, visitar_url, descargar_archivo, leer_archivo]
print("Herramientas listas:", [t.name for t in HERRAMIENTAS])

## Paso 6 — Crear el agente y responder una pregunta

La plantilla de pregunta incluye las **reglas oficiales de formato de GAIA** (por eso el exact-match no nos mata): números sin comas ni unidades, texto sin artículos, listas separadas por comas.

In [ ]:
from smolagents import CodeAgent, LiteLLMModel

PLANTILLA = """You are an AI assistant answering GAIA benchmark questions. Use tools (web search, visiting pages, reading attached files, Python calculations) to solve the question, verify the result, and then give ONLY the final answer.

Rules for the FINAL ANSWER (it is scored by exact string match - follow them strictly):
- Output ONLY the answer: a number, a short string, or a comma-separated list.
- Numbers: no thousands commas, no units, no currency symbols, no % sign unless asked.
- Strings: no articles (a, the), no abbreviations unless asked, write digits in plain text.
- Comma-separated lists: apply the rules above per element, keep the requested order.
- No explanation, no trailing period.

Question:
{question}

{extra}

Now solve it step by step with tools, verify, and produce the final answer in the required format."""

def construir_agente(model_id: str, api_key) -> "CodeAgent":
    kw = {"model_id": model_id, "max_tokens": 4096}
    if api_key:
        kw["api_key"] = api_key
    modelo = LiteLLMModel(**kw)

    def pausa_callback(*args, **kwargs):  # smolagents cambia cómo llama a este callback según la versión
        if model_id.startswith("gemini"):
            time.sleep(6)  # el tier gratis de Gemini tiene ~10-15 peticiones/min
    return CodeAgent(model=modelo, tools=list(HERRAMIENTAS),
                     max_steps=10, verbosity_level=1,
                     additional_authorized_imports=["os", "json", "base64"],
                     step_callbacks=[pausa_callback])

def responder(agente, pregunta: dict) -> str:
    tid = str(pregunta["task_id"])
    texto = pregunta.get("question", pregunta.get("Question", ""))
    extra = ""
    if tid in adjuntos:
        extra = f"Attached file saved locally at: {adjuntos[tid]} (read it with leer_archivo)."
    tarea = PLANTILLA.format(question=texto, extra=extra)
    try:
        respuesta = agente.run(task=tarea)
        return str(respuesta).strip().rstrip(".")
    except Exception as e:
        print(f"  [error en {tid[:8]}: {str(e)[:150]}]")
        return ""

print("Funciones listas: construir_agente(model_id, api_key) y responder(agente, pregunta)")

## Paso 7 — Prueba de modelos (5 preguntas × 3 candidatos)

Mide qué modelo rinde mejor **con tus claves**: Gemini (gratis), GPT-4o (pagada) y Qwen3-32B en Groq (gratis). Elige el ganador para las 20 completas. Si no tienes clave de Groq, borra esa línea del diccionario.

In [ ]:
CANDIDATOS = {
    "gemini-3.5-flash": ("gemini/gemini-3.5-flash", os.environ.get("GEMINI_API_KEY") or None),
    "gpt-4o":           ("openai/gpt-4o",          os.environ.get("OPENAI_API_KEY") or None),
    "qwen3-32b-groq":   ("groq/qwen3-32b",         os.environ.get("GROQ_API_KEY") or None),
}

MUESTRA = preguntas[:5]
resultados_bakeoff = {}

for nombre, (model_id, clave) in CANDIDATOS.items():
    if not clave:
        print(f"⏭️  {nombre}: sin clave, se salta.")
        continue
    print(f"\n===== {nombre} =====")
    agente = construir_agente(model_id, clave)
    aciertos = 0
    for q in MUESTRA:
        resp = responder(agente, q)
        gt = ground_truth.get(str(q["task_id"]), "")
        ok = resp.strip().lower() == gt.strip().lower()
        aciertos += ok
        print(f"  {'✅' if ok else '❌'} {str(q['task_id'])[:8]} → tú: {resp[:60]!r} | GT: {gt[:60]!r}")
    resultados_bakeoff[nombre] = aciertos / len(MUESTRA) * 100
    print(f"  → {aciertos}/{len(MUESTRA)} ({resultados_bakeoff[nombre]:.0f}%)")

print("\n===== RESUMEN =====")
for nombre, pct in resultados_bakeoff.items():
    print(f"{nombre:18s} {pct:5.1f}%")

## Paso 8 — Corre las 20 preguntas con el modelo ganador

Cambia `MODELO_GANADOR` según el resultado del Paso 7. Las respuestas se guardan en `respuestas.json` para no perder el progreso (si una petición falla, al reejecutar solo hace las que falten).

In [ ]:
MODELO_GANADOR = "gpt-4o"  # pon aquí el nombre del ganador del Paso 7
MODEL_ID_GANADOR, CLAVE_GANADOR = CANDIDATOS[MODELO_GANADOR]

respuestas = {}
if os.path.exists("respuestas.json"):
    with open("respuestas.json") as f:
        respuestas = json.load(f)
    print(f"Retomando: ya hay {len(respuestas)} respuestas guardadas.")

agente = construir_agente(MODEL_ID_GANADOR, CLAVE_GANADOR)

for i, q in enumerate(preguntas, 1):
    tid = str(q["task_id"])
    if respuestas.get(tid):
        continue
    print(f"\n--- Pregunta {i}/20 ---")
    respuestas[tid] = responder(agente, q)
    print("Respuesta:", respuestas[tid][:120])
    with open("respuestas.json", "w") as f:
        json.dump(respuestas, f, ensure_ascii=False, indent=2)

print(f"\nCompletadas {len(respuestas)}/20. Guardado en respuestas.json")

## Paso 9 — Nota en local (antes de enviar nada)

Comparamos con el ground truth del Paso 3 usando **la misma regla del servidor**: `strip().lower()` e igualdad exacta. Aquí es donde iteramos para subir del 30% al 70%.

In [ ]:
def normalizar(x: str) -> str:
    return str(x).strip().lower()

aciertos = 0
for q in preguntas:
    tid = str(q["task_id"])
    mia = normalizar(respuestas.get(tid, ""))
    gt  = normalizar(ground_truth.get(tid, ""))
    ok = bool(gt) and mia == gt
    aciertos += ok
    print(f"{'✅' if ok else '❌'} {tid[:8]}  tú: {mia[:55]!r:60s} GT: {gt[:55]!r}")

pct = aciertos / len(preguntas) * 100
print(f"\nNOTA LOCAL: {aciertos}/{len(preguntas)} = {pct:.0f}%  (certificado: 30% · objetivo: 70%)")

## Extra A — Corrección de formato (opcional, una vez)

El agente a veces acierta el contenido pero formatea mal la respuesta final (espacios en listas, abreviaturas). Esta celda corrige los casos conocidos sobre `respuestas.json`. Añade aquí los tuyos si detectas más en el Paso 9.

In [ ]:
import json

with open("respuestas.json") as f:
    respuestas = json.load(f)

CORRECCIONES = {
    "6f37996b": "b, e",                # faltaba espacio tras la coma
    "bda648d7": "saint petersburg",    # GAIA prohíbe abreviaturas (st. → saint)
}

for prefijo, valor in CORRECCIONES.items():
    for tid in respuestas:
        if tid.startswith(prefijo):
            respuestas[tid] = valor

with open("respuestas.json", "w") as f:
    json.dump(respuestas, f, ensure_ascii=False, indent=2)
print("Correcciones aplicadas. Reejecuta el Paso 9 para ver la nota.")

## Extra B — Re-correr solo las falladas

Tras arreglar los adjuntos (o mejorar la plantilla), borra de `respuestas.json` las preguntas que quieras reintentar y reejecuta el Paso 8: solo hará las que falten.

In [ ]:
import json

# Prefijos de las preguntas a reintentar (mira las ❌ del Paso 9)
A_REINTENTAR = ["99c9cc74", "f918266a", "7bd855d8", "1f975693"]

with open("respuestas.json") as f:
    respuestas = json.load(f)
for tid in list(respuestas):
    if tid[:8] in A_REINTENTAR:
        del respuestas[tid]
with open("respuestas.json", "w") as f:
    json.dump(respuestas, f, ensure_ascii=False, indent=2)
print("Borradas. Reejecuta el Paso 8 (solo hará las eliminadas).")

## Paso 10 — Enviar al leaderboard oficial

Solo cuando la nota local te convenza. `agent_code` es un enlace público a tu código: si no tienes Space PRO, usa tu repo de GitHub (lo acepta igual; lo importante es que sea un enlace público que muestre tu código).

In [ ]:
import requests, json

USERNAME   = "tu_usuario_de_hugging_face"
AGENT_CODE = "https://github.com/nicolasjanematas-crypto/agente-centinela"  # cámbialo si haces un repo propio para GAIA

answers = [{"task_id": str(q["task_id"]), "submitted_answer": respuestas.get(str(q["task_id"]), "")} for q in preguntas]

r = requests.post(API + "/submit", json={"username": USERNAME, "agent_code": AGENT_CODE, "answers": answers}, timeout=120)
print("HTTP", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

## Cómo subir del 30% al 70% (iteración)

1. Mira las ❌ del Paso 9 y clasifica el fallo: ¿buscó mal? ¿leyó mal un archivo? ¿formateó mal la respuesta (coma, unidades, mayúsculas)?
2. Si es formato: corrige el string en `respuestas.json` a mano (es lo que manda, no cómo lo obtuviste).
3. Si es de herramienta: mejora la herramienta o da más pistas en la plantilla (p. ej. "the data is in sheet 2").
4. Cambia de modelo solo si falla el razonamiento, no el formato.
5. Re-corre solo las falladas: borra su entrada de `respuestas.json` y reejecuta el Paso 8.

Suerte, y a por esas 14/20.